# download packages

In [ ]:
!pip install plotly

# Import the libraries

In [ ]:
import numpy as np
import pandas as pd

import io
import os
import tensorflow as tf

from PIL import Image
from glob import glob
import itertools

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns


import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, BatchNormalization, Dropout, Dense, MaxPool2D
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from IPython.display import display
#To see the value of multiple statements at once.
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# prepare train test splits

In [ ]:
def prepare_for_train_test(X, Y):
    # Splitting into train and test set
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=1)
    
    # Prepare data for training and testing the model
    train_datagen = ImageDataGenerator(rescale = 1./255,
                                  rotation_range = 10,
                                  width_shift_range = 0.2,
                                  height_shift_range = 0.2,
                                  shear_range = 0.2,
                                  horizontal_flip = True,
                                  vertical_flip = True,
                                  fill_mode = 'nearest')
    train_datagen.fit(X_train)
    test_datagen = ImageDataGenerator(rescale = 1./255)
    test_datagen.fit(X_test)
    return X_train, X_test, Y_train, Y_test

# Prepare model

In [ ]:
def create_model():
    model = Sequential()
    model.add(Conv2D(16, kernel_size = (3,3), input_shape = (28, 28, 3), activation = 'relu', padding = 'same'))
    model.add(MaxPool2D(pool_size = (2,2)))

    model.add(Conv2D(32, kernel_size = (3,3), activation = 'relu', padding = 'same'))
    model.add(MaxPool2D(pool_size = (2,2), padding = 'same'))

    model.add(Conv2D(64, kernel_size = (3,3), activation = 'relu', padding = 'same'))
    model.add(MaxPool2D(pool_size = (2,2), padding = 'same'))
    model.add(Conv2D(128, kernel_size = (3,3), activation = 'relu', padding = 'same'))
    model.add(MaxPool2D(pool_size = (2,2), padding = 'same'))

    model.add(Flatten())
    model.add(Dense(64, activation = 'relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(7, activation='softmax'))

    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001)

    model.compile(loss = 'sparse_categorical_crossentropy',
                 optimizer = optimizer,
                  metrics = ['accuracy'])
    print(model.summary())
    return model;


# prepare the training plan

In [ ]:
def train_model(model, X_train, Y_train, EPOCHS=25):
    early_stop = EarlyStopping(monitor='val_loss', patience=10, verbose=1, 
                           mode='auto')
                               #, restore_best_weights=True)
    
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, 
                              verbose=1, mode='auto')
    
    history = model.fit(X_train,
                        Y_train,
                        validation_split=0.2,
                        batch_size = 64,
                        epochs = EPOCHS,
                        callbacks = [reduce_lr, early_stop])
    return history

# test the model

In [ ]:
def test_model(model, X_test, Y_test):
    model_acc = model.evaluate(X_test, Y_test, verbose=0)[1]
    print("Test Accuracy: {:.3f}%".format(model_acc * 100))
    y_true = np.array(Y_test)
    y_pred = model.predict(X_test)
    y_pred = np.array(list(map(lambda x: np.argmax(x), y_pred)))
    clr = classification_report(y_true, y_pred, target_names=label_mapping.values())
    print(clr)
    
    sample_data = X_test[:15]
    plt.figure(figsize=(22, 12))
    for i in range(15):
        plt.subplot(3, 5, i + 1)
        plt.imshow(sample_data[i])
        plt.title(label_mapping[y_true[i][0]] + '|' + label_mapping[y_pred[i]])
        plt.axis("off")
    plt.show() 

# training curves plot

In [ ]:
def plot_model_training_curve(history):
    fig = make_subplots(rows=1, cols=2, subplot_titles=['Model Accuracy', 'Model Loss'])
    fig.add_trace(
        go.Scatter(
            y=history.history['accuracy'], 
            name='train_acc'), 
        row=1, col=1)
    fig.add_trace(
        go.Scatter(
            y=history.history['val_accuracy'], 
            name='val_acc'), 
        row=1, col=1)
    fig.add_trace(
        go.Scatter(
            y=history.history['loss'], 
            name='train_loss'), 
        row=1, col=2)
    fig.add_trace(
        go.Scatter(
            y=history.history['val_loss'], 
            name='val_loss'), 
        row=1, col=2)
    fig.show()

# multi class confusion matrix

In [ ]:
def cal_true_pred_classes(model, x_test, y_test):
    y_predict = model.predict(x_test)
    y_predict_classes = np.argmax(y_predict, axis=1)
    y_true_classes = np.array(y_test).flatten()
    return y_predict_classes, y_true_classes

def create_confusion_matrix(model, x_test_normalized, y_test, cm_plot_labels, name):
    # Get the true and predicted labels
    y_predict_classes, y_true_classes = cal_true_pred_classes(model, x_test_normalized, y_test)

    # Compute the confusion matrix
    confusion_matrix_computed = confusion_matrix(y_true_classes, y_predict_classes)

    # Plot the confusion matrix
    plot_confusion_matrix(confusion_matrix_computed, cm_plot_labels, name)


In [ ]:
def plot_confusion_matrix(cm, classes,
                          name,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    
    plt.figure(figsize=(8,6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(name)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.ylabel('True Labels')
    plt.xlabel('Predicted Labels')
    plt.savefig(f'{name}_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()


# read the data

In [ ]:
# Search for HAM10000 dataset root directory
possible_roots = [
    '/kaggle/input/skin-cancer-mnist-ham10000',
    '../input/skin-cancer-mnist-ham10000',
    '../input',
    './skin-cancer-mnist-ham10000',
    '.'
]
base_skin_dir = None
for r in possible_roots:
    if os.path.exists(os.path.join(r, 'HAM10000_metadata.csv')):
        base_skin_dir = r
        break

if base_skin_dir is None:
    for root, dirs, files in os.walk('/kaggle/input' if os.path.exists('/kaggle/input') else '..'):
        if 'HAM10000_metadata.csv' in files:
            base_skin_dir = root
            break

if base_skin_dir is None:
    base_skin_dir = '/kaggle/input/skin-cancer-mnist-ham10000'

print(f"Base dataset directory: {base_skin_dir}")

# Merging images from folders into one dictionary
all_image_paths = glob(os.path.join(base_skin_dir, '**', '*.jpg'), recursive=True)
if not all_image_paths and os.path.exists('/kaggle/input'):
    all_image_paths = glob(os.path.join('/kaggle/input', '**', '*.jpg'), recursive=True)

imageid_path_dict = {os.path.splitext(os.path.basename(x))[0]: x for x in all_image_paths}
print(f"Total images located: {len(imageid_path_dict)}")

lesion_type_dict = {
    'nv': 'Melanocytic nevi (nv)',
    'mel': 'Melanoma (mel)',
    'bkl': 'Benign keratosis-like lesions (bkl)',
    'bcc': 'Basal cell carcinoma (bcc)',
    'akiec': 'Actinic keratoses (akiec)',
    'vasc': 'Vascular lesions (vasc)',
    'df': 'Dermatofibroma (df)'
}
label_mapping = {
    0: 'nv',
    1: 'mel',
    2: 'bkl',
    3: 'bcc',
    4: 'akiec',
    5: 'vasc',
    6: 'df'
}
reverse_label_mapping = dict((value, key) for key, value in label_mapping.items())


In [ ]:
data = pd.read_csv(os.path.join(base_skin_dir, 'HAM10000_metadata.csv'))
print(f"Metadata shape: {data.shape}")
data.head()


In [ ]:
# Now lets see the sample of tile_df to look on newly made columns
data.head()

In [ ]:
data.describe(exclude=[np.number])

In [ ]:
data.isnull().any().sum()

# Imputing the Null values and preparing the data

In [ ]:
# Handling null values
data['age'] = data['age'].fillna(int(data['age'].mean())).astype('int32')


In [ ]:
# Adding cell_type and image_path columns
data['cell_type'] = data['dx'].map(lesion_type_dict.get)
data['path'] = data['image_id'].map(imageid_path_dict.get)

In [ ]:
# Adding image pixels
data['image_pixel'] = data['path'].map(lambda x: np.asarray(Image.open(x).resize((28,28))))

In [ ]:
# Displaying 2 sample images for each lesion type
sample_data = data.groupby('dx').head(2).reset_index(drop=True)
plt.figure(figsize=(20, 8))
for i in range(min(14, len(sample_data))):
    plt.subplot(2, 7, i + 1)
    plt.imshow(np.squeeze(sample_data['image_pixel'][i]))
    img_label = sample_data['cell_type'][i]
    plt.title(img_label, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
data['label'] = data['dx'].map(reverse_label_mapping.get)

data = data.sort_values('label')
data = data.reset_index()

In [ ]:
counter = 0
frames = [data]
for i in [4, 4, 11, 17, 45, 52]:
    counter += 1
    index = data[data['label'] == counter].index.values
    df_index = data.iloc[int(min(index)):int(max(index) + 1)]
    augmented_slice = pd.concat([df_index] * (i + 1), ignore_index=True)
    frames.append(augmented_slice)


In [ ]:
final_data = pd.concat(frames)

In [ ]:
print(data.shape)
print(final_data.shape)

In [ ]:
# ORIGINAL DATA
# Converting image pixel columnm into required format
X_orig = data['image_pixel'].to_numpy()
X_orig = np.stack(X_orig, axis=0)
Y_orig = np.array(data.iloc[:, -1:])
print(X_orig.shape)
print(Y_orig.shape)

In [ ]:
# AUGMENTED DATA
# Converting image pixel columnm into required format
X_aug = final_data['image_pixel'].to_numpy()
X_aug = np.stack(X_aug, axis=0)
Y_aug = np.array(final_data.iloc[:, -1:])
print(X_aug.shape)
print(Y_aug.shape)

In [ ]:
# For Original Dataset
X_train_orig, X_test_orig, Y_train_orig, Y_test_orig = prepare_for_train_test(X_orig, Y_orig)

In [ ]:
model =create_model()

In [ ]:
X_train_aug, X_test_aug, Y_train_aug, Y_test_aug = prepare_for_train_test(X_aug, Y_aug)

In [ ]:
# Train model with 25 epochs
EPOCHS = 25
model2_history = train_model(model, X_train_aug, Y_train_aug, EPOCHS=EPOCHS)


### 📊 Training & Validation Curves


In [ ]:
# Plot accuracy and loss curves
plot_model_training_curve(model2_history)


### 🧪 Model Evaluation on Test Set


In [ ]:
# Evaluate test set accuracy and display classification report
test_model(model, X_test_aug, Y_test_aug)


### 🔍 Confusion Matrix


In [ ]:
# Compute and plot confusion matrix
create_confusion_matrix(model, X_test_aug, Y_test_aug, list(label_mapping.values()), "Skin_Cancer")


### 📦 Export Model & MedSynapse Artifacts


In [ ]:
import json
import zipfile
from IPython.display import FileLink, display

# 1. Save model formats
model.save('skin_cancer_model.keras')
model.save('skin_cancer_model.h5')
model.save('Skin_Cancer.sav')
model.save_weights('Skin_Cancer.hdf5')

# 2. Compute final evaluation metrics
test_loss, test_acc = model.evaluate(X_test_aug, Y_test_aug, verbose=0)
y_pred_probs = model.predict(X_test_aug)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.array(Y_test_aug).flatten()
target_names = [label_mapping[i] for i in range(len(label_mapping))]

report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)

metrics = {
    "model_name": "Skin Cancer CNN (HAM10000)",
    "dataset": "kmader/skin-cancer-mnist-ham10000",
    "epochs_trained": len(model2_history.history['loss']),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "precision_macro": float(report['macro avg']['precision']),
    "recall_macro": float(report['macro avg']['recall']),
    "f1_macro": float(report['macro avg']['f1-score']),
    "classification_report": report
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

# 3. Save class metadata
classes_meta = {
    "classes": target_names,
    "label_mapping": label_mapping,
    "reverse_label_mapping": reverse_label_mapping,
    "lesion_type_dict": lesion_type_dict,
    "input_shape": [28, 28, 3]
}

with open('classes.json', 'w') as f:
    json.dump(classes_meta, f, indent=4)

# 4. Package artifacts bundle
artifact_files = [
    'skin_cancer_model.keras',
    'skin_cancer_model.h5',
    'Skin_Cancer.hdf5',
    'classes.json',
    'metrics.json'
]
if os.path.exists('Skin_Cancer_confusion_matrix.png'):
    artifact_files.append('Skin_Cancer_confusion_matrix.png')

with zipfile.ZipFile('skin_cancer_artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for f in artifact_files:
        if os.path.exists(f):
            zipf.write(f)

print('=' * 60)
print('🎉 Model Training and Packaging Complete!')
print(f'📊 Test Accuracy: {test_acc * 100:.2f}%')
print(f'📊 Test Loss: {test_loss:.4f}')
print(f'📊 Macro F1-Score: {metrics["f1_macro"] * 100:.2f}%')
print("📦 Artifacts packaged into 'skin_cancer_artifacts.zip'")
print('=' * 60)
display(FileLink('skin_cancer_artifacts.zip'))
